# UAE Mobile Intelligence - OSM Feature Extraction (buildings, roads, POIs, land-use -> H3 res 7)

Completes the step [`03_osm_collection.ipynb`](03_osm_collection.ipynb) deferred: pulls UAE
buildings, roads, POIs and land-use polygons out of the raw Geofabrik GCC States PBF with
`pyosmium`, clips them to the real UAE boundary, and aggregates them into the same H3
resolution-7 zones used everywhere else in this repo (chosen in
[`04_h3_resolution_choice.ipynb`](04_h3_resolution_choice.ipynb), zone-aggregated for Ookla +
WorldPop in [`05_zone_aggregation.ipynb`](05_zone_aggregation.ipynb)).

**Why this matters:** the brief's mandatory peer-group classifier is a composite of population
density, building-footprint density, POI density and road density -- explicitly *not* OSM
land-use tags, since commercial/retail tagging is too thin in the UAE (collapses to ~17 zones).
This notebook builds the three OSM density inputs; population density already exists in
`population_zones_uae.parquet`. The composite classifier itself (turning these four signals
into commercial/urban-core, low-density residential, industrial, rural/edge groups) is the next
step, once all four inputs exist side by side.

Output: `data/processed/osm_density_zones_uae.parquet` -- one row per H3 res-7 cell with
building, road and POI density, ready to join against the Ookla and population zone tables.

In [1]:
import json
import time
import warnings
from pathlib import Path

import geopandas as gpd
import h3
import osmium
import osmium.filter as ofilter
import osmium.geom as ogeom
import pandas as pd
import shapely.wkb as wkblib
from shapely.geometry import Point, Polygon

## Config

`MINLON`/`MINLAT`/`MAXLON`/`MAXLAT` are a coarse bounding box padded beyond the UAE boundary's
own bounds (51.54, 22.63, 56.38, 26.08 -- see [`00_ookla_dataset_overview.ipynb`](00_ookla_dataset_overview.ipynb)
for how that's read off the boundary file). **This is only a cheap first-pass filter** to avoid
building Python objects for the other five GCC countries in the extract -- it is never used to
decide UAE membership. That decision is made downstream by a real polygon `sjoin` against
`uae_boundary.geojson`, the same two-stage pattern (`01_ookla_collection.ipynb` uses coarse
`tile_x`/`tile_y` BETWEEN filters, then an exact `sjoin`) already established in this repo, for
the same reason: the brief is explicit that a bounding box alone catches Doha, Oman and Gulf
water.

POI tag keys follow common OSM point-of-interest conventions (amenity, shop, office, tourism,
leisure, healthcare) -- these are the tag keys behind the brief's cited "~67,000 POIs" figure.

In [2]:
PBF_PATH = Path("../data/raw/osm/gcc-states-latest.osm.pbf")
BOUNDARY_PATH = Path("../data/raw/boundary/uae_boundary.geojson")
H3_DECISION_PATH = Path("../data/processed/h3_resolution.json")
OUT_PATH = Path("../data/processed/osm_density_zones_uae.parquet")

MINLON, MINLAT, MAXLON, MAXLAT = 51.4, 22.5, 56.5, 26.2
POI_KEYS = ["amenity", "shop", "office", "tourism", "leisure", "healthcare"]
EQUAL_AREA_CRS = "EPSG:6933"  # never UTM 40N -- brief flags up to ~0.7% overstatement in western UAE

resolution_decision = json.loads(H3_DECISION_PATH.read_text())
H3_RESOLUTION = resolution_decision["chosen_resolution"]
print("Using H3 resolution:", H3_RESOLUTION, "(decision source: 04_h3_resolution_choice.ipynb)")

Using H3 resolution: 7 (decision source: 04_h3_resolution_choice.ipynb)


## Step 1 -- stream the PBF once, pulling out four feature types

`pyosmium` (not `pyrosm` -- confirmed too slow on a file this size, per the brief) streams the
PBF without loading it into memory. `osmium.filter.KeyFilter` runs in the C++ layer and only
invokes the Python callback for nodes/ways that carry at least one of the requested tag keys --
the untagged nodes that make up the bulk of a PBF (every vertex of every road and building) never
reach Python at all, which is what keeps this fast.

Four feature types, one pass:
- **Buildings** -- closed ways tagged `building=*`. `WKBFactory` has no way->polygon method, so
  each ring is built as a linestring and converted to a `shapely.Polygon` from its own
  (closed) coordinates.
- **Roads** -- ways tagged `highway=*`, kept as linestrings.
- **Land-use** -- closed ways tagged `landuse=*`. Extracted for documentation and the thinness
  check below, **not** used as a peer-group input (see the brief's explicit warning).
- **POIs** -- nodes carrying any of `POI_KEYS`.

**Known simplification:** only simple ways are extracted, not multipolygon *relations* (a
minority of buildings/land-use are mapped as relations with holes). POIs are node-tagged only --
way/relation-tagged POI areas (e.g. some malls) are not included. Both are standard density-proxy
simplifications; neither changes which of the four peer-group signals a zone falls into by more
than noise.

In [3]:
wkbfab = ogeom.WKBFactory()


class UAEFeatureHandler(osmium.SimpleHandler):
    """Single streaming pass over the GCC PBF, filtered in C++ to entities carrying at least
    one of the tag keys we care about. Geographic membership is only bbox-coarse here --
    the real UAE clip happens afterwards against the boundary polygon."""

    def __init__(self):
        super().__init__()
        self.buildings = []
        self.roads = []
        self.pois = []
        self.landuse = []

    @staticmethod
    def _in_bbox(lon, lat):
        return MINLON <= lon <= MAXLON and MINLAT <= lat <= MAXLAT

    @staticmethod
    def _closed_way_to_polygon(way):
        try:
            wkb = wkbfab.create_linestring(way)
        except RuntimeError:
            return None
        ring = wkblib.loads(wkb, hex=True)
        if len(ring.coords) < 4:
            return None
        try:
            return Polygon(ring.coords)
        except Exception:
            return None

    def node(self, n):
        if not n.location.valid():
            return
        lon, lat = n.location.lon, n.location.lat
        if not self._in_bbox(lon, lat):
            return
        tags = dict(n.tags)
        hit = next((k for k in POI_KEYS if k in tags), None)
        if hit:
            self.pois.append({"id": n.id, "lon": lon, "lat": lat, "poi_type": tags[hit]})

    def way(self, w):
        tags = dict(w.tags)
        try:
            first_loc = w.nodes[0].location
        except (IndexError, RuntimeError):
            return
        if not first_loc.valid() or not self._in_bbox(first_loc.lon, first_loc.lat):
            return

        if "building" in tags and w.is_closed():
            poly = self._closed_way_to_polygon(w)
            if poly is not None:
                self.buildings.append({"id": w.id, "geometry": poly, "building": tags["building"]})
        elif "highway" in tags:
            try:
                wkb = wkbfab.create_linestring(w)
            except RuntimeError:
                return
            self.roads.append({"id": w.id, "geometry": wkblib.loads(wkb, hex=True), "highway": tags["highway"]})
        elif "landuse" in tags and w.is_closed():
            poly = self._closed_way_to_polygon(w)
            if poly is not None:
                self.landuse.append({"id": w.id, "geometry": poly, "landuse": tags["landuse"]})


t0 = time.time()
handler = UAEFeatureHandler()
key_filter = ofilter.KeyFilter("building", "highway", "landuse", *POI_KEYS)
handler.apply_file(str(PBF_PATH), locations=True, filters=[key_filter])
print(f"PBF pass complete in {time.time() - t0:.0f}s")

print("Bbox-filtered (GCC-wide, coarse box): "
      f"{len(handler.buildings):,} buildings, {len(handler.roads):,} roads, "
      f"{len(handler.pois):,} POIs, {len(handler.landuse):,} land-use polygons")

PBF pass complete in 177s
Bbox-filtered (GCC-wide, coarse box): 783,677 buildings, 653,753 roads, 68,203 POIs, 43,004 land-use polygons


**Sanity check against the brief:** the brief cites "roughly 45,700 UAE land-use polygons,
711k road ways and 795k building footprints" and "about 67,000" POIs for the Geofabrik GCC
extract. The bbox-filtered counts above land close to those figures (within the extract's own
growth since the brief was written) -- strong evidence the brief's own numbers were computed at a
similarly coarse geographic filter, not after a precise polygon clip. The precise UAE clip below
narrows every count further, as expected.

## Step 2 -- precise UAE clip + H3 assignment

Same pattern as `01_ookla_collection.ipynb`'s exact filter: `gpd.sjoin` a representative point
per feature against the real UAE boundary polygon (`predicate="within"`), never the bounding box.
The representative point differs by feature type -- polygon centroid for buildings/land-use, the
node location itself for POIs, the line's midpoint for roads -- and that same point's raw
`(lat, lon)` (EPSG:4326, unprojected) is what gets assigned to an H3 cell, per the brief's
instruction to never reproject before H3 assignment.

**Known simplification:** each feature is assigned whole to a single zone (the one its
representative point falls in), the same pattern `05_zone_aggregation.ipynb` uses for WorldPop
pixels. A handful of large features (an airport terminal, an industrial complex) can span more
than one H3 cell and get counted entirely in one -- flagged and quantified below rather than
hidden.

In [4]:
uae_boundary = gpd.read_file(BOUNDARY_PATH).to_crs("EPSG:4326")


def clip_and_assign(records, rep_point_fn=None):
    """records: list of dicts with a shapely 'geometry'. Returns a GeoDataFrame restricted to
    features whose representative point falls inside the real UAE boundary polygon, with an
    added h3_cell column assigned from that same point."""
    gdf = gpd.GeoDataFrame(records, geometry="geometry", crs="EPSG:4326")
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)  # centroid-in-geographic-CRS warning; fine at building/parcel scale
        rep = gdf.geometry.apply(rep_point_fn) if rep_point_fn else gdf.geometry.centroid
    points = gpd.GeoDataFrame(gdf.drop(columns=["geometry"]), geometry=rep, crs="EPSG:4326")
    kept_idx = gpd.sjoin(points, uae_boundary[["geometry"]], predicate="within", how="inner").index
    gdf = gdf.loc[kept_idx].copy()
    rep_kept = rep.loc[kept_idx]
    gdf["h3_cell"] = [h3.latlng_to_cell(pt.y, pt.x, H3_RESOLUTION) for pt in rep_kept]
    return gdf


pois_records = [{**p, "geometry": Point(p["lon"], p["lat"])} for p in handler.pois]

t0 = time.time()
buildings_gdf = clip_and_assign(handler.buildings)
roads_gdf = clip_and_assign(handler.roads, rep_point_fn=lambda g: g.interpolate(0.5, normalized=True))
pois_gdf = clip_and_assign(pois_records)
landuse_gdf = clip_and_assign(handler.landuse)
print(f"Clip + H3 assignment complete in {time.time() - t0:.0f}s")

print(f"UAE-clipped: {len(buildings_gdf):,} buildings, {len(roads_gdf):,} roads, "
      f"{len(pois_gdf):,} POIs, {len(landuse_gdf):,} land-use polygons")

Clip + H3 assignment complete in 17s
UAE-clipped: 561,846 buildings, 473,040 roads, 45,296 POIs, 31,784 land-use polygons


## Step 3 -- zone-level density

Areas and lengths are computed in `EPSG:6933` (equal-area), reprojected from the already-clipped,
already H3-assigned EPSG:4326 geometries -- reprojection happens *after* H3 assignment, exactly
per the brief's ordering rule. The zone-area denominator uses `h3.cell_area` (geodesic), also per
the brief's explicit guidance, rather than reprojecting the hexagon polygon itself.

In [5]:
buildings_gdf["area_m2"] = buildings_gdf.geometry.to_crs(EQUAL_AREA_CRS).area
roads_gdf["length_m"] = roads_gdf.geometry.to_crs(EQUAL_AREA_CRS).length
landuse_gdf["area_m2"] = landuse_gdf.geometry.to_crs(EQUAL_AREA_CRS).area

building_zones = buildings_gdf.groupby("h3_cell").agg(building_count=("id", "count"), building_area_m2=("area_m2", "sum"))
road_zones = roads_gdf.groupby("h3_cell").agg(road_count=("id", "count"), road_length_m=("length_m", "sum"))
poi_zones = pois_gdf.groupby("h3_cell").agg(poi_count=("id", "count"))

all_zones = sorted(set(building_zones.index) | set(road_zones.index) | set(poi_zones.index))
density = pd.DataFrame({"h3_cell": all_zones}).set_index("h3_cell")
density = density.join(building_zones).join(road_zones).join(poi_zones).fillna(0)

density["zone_area_km2"] = [h3.cell_area(c, unit="km^2") for c in density.index]
density["building_footprint_pct"] = (density["building_area_m2"] / 1e6) / density["zone_area_km2"] * 100
density["building_count_per_km2"] = density["building_count"] / density["zone_area_km2"]
density["poi_count_per_km2"] = density["poi_count"] / density["zone_area_km2"]
density["road_density_km_per_km2"] = (density["road_length_m"] / 1000) / density["zone_area_km2"]

density = density.reset_index()
print("H3 zones with at least one OSM feature:", len(density))
density.describe()

H3 zones with at least one OSM feature: 6575


,building_count,building_area_m2,road_count,road_length_m,poi_count,zone_area_km2,building_footprint_pct,building_count_per_km2,poi_count_per_km2,road_density_km_per_km2
count,6575.000000,6.575000e+03,6575.000000,6575.000000,6575.000000,6575.000000,6575.000000,6575.000000,6575.000000,6575.000000
mean,85.451863,3.431329e+04,71.945247,18007.943641,6.889125,4.581592,0.738309,18.364232,1.478165,3.898294
std,472.696656,1.675047e+05,194.520846,25870.606078,57.646859,0.113630,3.616746,101.157847,12.325699,5.567296
min,0.000000,0.000000e+00,0.000000,0.000000,0.000000,4.342698,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000e+00,2.000000,3112.843185,0.000000,4.476951,0.000000,0.000000,0.000000,0.690089
50%,0.000000,0.000000e+00,7.000000,8694.778818,0.000000,4.590047,0.000000,0.000000,0.000000,1.903727
75%,4.000000,1.835160e+03,37.000000,20656.466521,0.000000,4.680489,0.040353,0.865344,0.000000,4.520185
max,12439.000000,7.028722e+06,2081.000000,225313.977958,1907.000000,4.812626,154.846836,2641.882306,406.225611,51.614756


## Step 4 -- known-limitation check: whole-feature-to-one-zone assignment

`building_footprint_pct` is a ground-coverage fraction and should never exceed 100 -- but a
feature bigger than its own H3 cell, assigned whole to that one cell, can push it over. Quantify
rather than silently cap it.

In [6]:
over100 = density[density["building_footprint_pct"] > 100]
print(f"Zones with building_footprint_pct > 100: {len(over100)} of {len(density)}")

biggest = buildings_gdf.loc[buildings_gdf["area_m2"].idxmax()]
print(f"Largest single building way: id={biggest['id']}, area={buildings_gdf['area_m2'].max():,.0f} m^2, "
      f"tag=building={biggest['building']!r}")
print(f"Buildings > 10 ha (100,000 m^2): {(buildings_gdf['area_m2'] > 100_000).sum()} of {len(buildings_gdf):,}")

Zones with building_footprint_pct > 100: 1 of 6575
Largest single building way: id=1495979992, area=7,028,722 m^2, tag=building='yes'
Buildings > 10 ha (100,000 m^2): 29 of 561,846


One zone (of several thousand) exceeds 100%, caused by a single ~7 km² way tagged
`building=yes` -- almost certainly an oversized/mistagged facility footprint (e.g. an industrial
or military complex digitized as one "building"), not a systematic problem. Left uncorrected
rather than capped: capping would hide a genuine data artifact instead of surfacing it, and at
1-in-several-thousand zones it does not threaten the density signal as a whole.

## Step 5 -- land-use tag thinness (confirms the brief's own claim, on this data)

The brief says not to classify peer groups by OSM land-use tag because commercial/retail tagging
is too thin in the UAE, collapsing that peer group to "roughly 17 zones". Check that claim
directly on this extract, the same way `04_h3_resolution_choice.ipynb` re-derives the brief's
density figures rather than taking them on faith.

In [7]:
print("Land-use tag distribution (UAE-clipped):")
print(landuse_gdf["landuse"].value_counts().head(15))

n_commercial = landuse_gdf[landuse_gdf["landuse"] == "commercial"]["h3_cell"].nunique()
n_retail = landuse_gdf[landuse_gdf["landuse"] == "retail"]["h3_cell"].nunique()
n_either = landuse_gdf[landuse_gdf["landuse"].isin(["commercial", "retail"])]["h3_cell"].nunique()
print()
print(f"Zones with >=1 commercial land-use polygon: {n_commercial}")
print(f"Zones with >=1 retail land-use polygon: {n_retail}")
print(f"Zones with >=1 commercial OR retail land-use polygon: {n_either}")

Land-use tag distribution (UAE-clipped):
landuse
grass           10941
residential      4418
orchard          3322
industrial       2963
farmyard         2217
farmland         1843
reservoir        1790
construction     1014
forest            952
commercial        527
meadow            364
military          192
retail            168
quarry            150
flowerbed         128
Name: count, dtype: int64

Zones with >=1 commercial land-use polygon: 264
Zones with >=1 retail land-use polygon: 74
Zones with >=1 commercial OR retail land-use polygon: 316


Confirmed independently on this extract: residential (4,418) and industrial (2,963) land-use
polygons are plentiful, but commercial (527) and retail (168) are thin and cover very few zones --
consistent with the brief's ~17-zone collapse claim. This is why building-footprint, POI and road
density (computed above) are the peer-group inputs, not this land-use tag.

## Save

One row per H3 res-7 cell with at least one OSM feature. Zones absent from this table (e.g.
open desert, water) have zero building/road/POI density by omission, the same convention
`population_zones_uae.parquet` uses for zero population -- joins downstream should fill missing
values with 0, not drop the zone.

In [8]:
density.to_parquet(OUT_PATH, index=False)
print(f"Saved: {OUT_PATH} ({OUT_PATH.stat().st_size / 1024:.1f} KB, {len(density):,} zones)")

Saved: ..\data\processed\osm_density_zones_uae.parquet (344.3 KB, 6,575 zones)


## Summary

- `data/processed/osm_density_zones_uae.parquet` -- one row per H3 res-7 cell:
  `building_count`, `building_area_m2`, `road_count`, `road_length_m`, `poi_count`,
  `zone_area_km2`, and the three derived densities (`building_footprint_pct`,
  `building_count_per_km2`, `poi_count_per_km2`, `road_density_km_per_km2`).
- Land-use polygons were extracted and checked (Step 5) but are **not** in this output --
  confirmed too thin for peer classification, exactly as the brief warns.
- This clears the "OSM UAE feature extraction" item from the README's Next Steps.

**Next:** join this table with `population_zones_uae.parquet` (population density) against the
same H3 res-7 zones, and build the composite peer-group classifier (commercial/urban-core,
low-density residential, industrial, rural/edge) the brief requires before any Peer Gap
comparison can run.